# Prompt playground

Standalone prompt/debug console for the LMEnt checkpoints: **a string goes in, the
model's continuation comes out**. Nothing here imports the repo — only `torch` +
`transformers` — so the notebook runs anywhere the model can be downloaded.

Run cell 1 (load), then cell 2 (`generate`), then edit and rerun the cells below.

Knobs on `generate`: `step` (which checkpoint answers), `temperature` (0 = greedy,
deterministic), `top_p`, `top_k`, `max_new_tokens`, `repetition_penalty`, `seed`,
`stop`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'dhgottesman/LMEnt-1B-6E'     # 1B, single epoch
STEP = 658032                            # step0, step10000 ... step650000, step658032 (final)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if DEVICE == 'cuda' else torch.float32

# The tokenizer is identical at every checkpoint, so it is loaded once.
tok = AutoTokenizer.from_pretrained(MODEL_ID, subfolder=f'step{STEP}')

_models = {}


def load(step=None):
    """The model at one checkpoint, downloaded once and then kept in memory."""
    step = STEP if step is None else step
    if step not in _models:
        print(f'loading step{step} on {DEVICE} ...', flush=True)
        _models[step] = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, subfolder=f'step{step}', dtype=DTYPE).to(DEVICE).eval()
        print('ready')
    return _models[step]


load()      # warm up the default checkpoint

In [ ]:
@torch.no_grad()
def generate(prompt, step=None, max_new_tokens=15, temperature=0.0, top_p=1.0,
             top_k=0, repetition_penalty=1.0, seed=0, stop=('\n\n',)):
    """Continue `prompt` and return the continuation as a string.

    temperature=0 -> greedy (deterministic, what the eval uses); > 0 -> sampling,
    where top_p / top_k / seed matter (seed=None for fresh randomness each call).
    stop: cut the output at the first of these strings (None keeps everything).
    """
    model = load(step)
    enc = tok(prompt, return_tensors='pt').to(model.device)
    kw = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id,
              repetition_penalty=repetition_penalty)
    if temperature > 0:
        if seed is not None:
            torch.manual_seed(seed)
        kw.update(do_sample=True, temperature=temperature, top_p=top_p,
                  top_k=top_k or 0)
    else:
        kw.update(do_sample=False)
    out = model.generate(**enc, **kw)
    text = tok.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True)
    for s in stop or ():
        text = text.split(s)[0]
    return text

## Try it

The eval prompts are cloze-style — `The occupation of X is`, `The author of X is`,
`The capital of X is`, `X is the capital of` — optionally with demonstrations in front,
separated by blank lines. Paste anything you like.

In [ ]:
print(generate('The capital of France is'))

In [ ]:
# A few-shot prompt: demonstrations, a blank line between each, then the query.
prompt = """The occupation of Kanye West is record producer

The occupation of Claudio Naranjo is psychiatrist

The occupation of George Rankin is"""

print(generate(prompt))

In [ ]:
# Knobs: longer output, sampling, a different checkpoint.
p = 'The genre of Kind of Blue is'
print('greedy    :', repr(generate(p)))
print('longer    :', repr(generate(p, max_new_tokens=40, stop=None)))
print('sampled   :', [generate(p, temperature=0.8, seed=s) for s in (1, 2, 3)])
print('step100000:', repr(generate(p, step=100000)))